### Import dependancies

In [1]:
import cv2
import ultralytics
import os
import PIL
import numpy as np
import math
import time
import mailtrap as mt
import requests
import torch
import math

from ultralytics import YOLO
from yolov6.utils.events import LOGGER, load_yaml
from yolov6.layers.common import DetectBackend
from yolov6.data.data_augment import letterbox
from yolov6.utils.nms import non_max_suppression
from yolov6.core.inferer import Inferer
from typing import List, Optional

ModuleNotFoundError: No module named 'cv2'

### Capture bed calibration image

In [ ]:

class PhotoCaptureApp:
    def __init__(self, save_folder):
        self.save_folder = save_folder

        # Set up OpenCV to capture video
        self.cap = cv2.VideoCapture(0)

        while not self.cap.isOpened():
            pass

        # Set the window size
        self.cap.set(3, 640)
        self.cap.set(4, 480)

        # Define font and color
        self.font = cv2.FONT_HERSHEY_SIMPLEX
        self.font_size = 0.8
        self.font_thickness = 2
        self.font_color = (57, 255, 20)  # Neon Green
        self.font_outline_thickness = 3
        self.font_outline_color = (0, 0, 0)

    def update(self):
        ret, frame = self.cap.read()

        if ret:
            # Convert BGR image to RGB
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Add text to the image
            text_line1 = "Capture a person-free, well-framed"
            text_line2 = "bed calibration photo by pressing 'c'."
            text = f"{text_line1}\n{text_line2}"

            text_position = (10, 20)  # Adjusted position

            cv2.putText(img, text_line1, (text_position[0], text_position[1] + 30), self.font, self.font_size,
                        self.font_outline_color, self.font_outline_thickness, cv2.LINE_AA)
            cv2.putText(img, text_line2, (text_position[0], text_position[1] + 60), self.font, self.font_size,
                        self.font_outline_color, self.font_outline_thickness, cv2.LINE_AA)

            cv2.putText(img, text_line1, (text_position[0], text_position[1] + 30), self.font, self.font_size,
                        self.font_color, self.font_thickness, cv2.LINE_AA)
            cv2.putText(img, text_line2, (text_position[0], text_position[1] + 60), self.font, self.font_size,
                        self.font_color, self.font_thickness, cv2.LINE_AA)

            # Save the image without overlay text directly to the specified folder
            save_path = os.path.join(self.save_folder, "calibration_image.jpg")
            cv2.imwrite(save_path, frame)

            # Display the image in a separate OpenCV window (converted back to BGR for display)
            cv2.imshow("Photo Capture", cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

    def run(self):
        while True:
            self.update()
            key = cv2.waitKey(1) & 0xFF
            if key == ord('c'):
                # Capture the current frame, save it, and exit
                ret, frame = self.cap.read()
                if ret:
                    # Save the image directly to the specified folder
                    save_path = os.path.join(self.save_folder, "calibration_image.jpg")
                    cv2.imwrite(save_path, frame)
                    print(f"Image saved to {save_path}")
                    break

        # Release the camera and close OpenCV windows
        self.cap.release()
        cv2.destroyAllWindows()

if __name__ == "__main__":
    save_folder = "/Users/emibetroldan/Downloads/Science_Fair/fotos_out/crop_out/bed_calibration/"
    app = PhotoCaptureApp(save_folder)
    app.run()


### Process bed calibration image

In [ ]:
#@title Set-up model. { run: "auto" }
checkpoint:str ="yolov6n" #@param ["yolov6s", "yolov6n", "yolov6t"]
device:str = "cpu"#@param ["gpu", "cpu"]
half:bool = False #@param {type:"boolean"}

#Change directory so that imports wortk correctly
if os.getcwd()=="/content":
  os.chdir("YOLOv6")

# Set the local image path
image_path = "/Users/emibetroldan/Downloads/Science_Fair/fotos_out/crop_out/bed_calibration/calibration_image.jpg"

# Download weights
if not os.path.exists(f"{checkpoint}.pt"):
  print("Downloading checkpoint...")
  os.system(f"""wget -c https://github.com/meituan/YOLOv6/releases/download/0.4.0/{checkpoint}.pt""")

# Set-up hardware options
cuda = device != 'cpu' and torch.cuda.is_available()
device = torch.device('cuda:0' if cuda else 'cpu')

def check_img_size(img_size, s=32, floor=0):
  def make_divisible( x, divisor):
    # Upward revision the value x to make it evenly divisible by the divisor.
    return math.ceil(x / divisor) * divisor
  """Make sure image size is a multiple of stride s in each dimension, and return a new shape list of image."""
  if isinstance(img_size, int):  # integer i.e. img_size=640
      new_size = max(make_divisible(img_size, int(s)), floor)
  elif isinstance(img_size, list):  # list i.e. img_size=[640, 480]
      new_size = [max(make_divisible(x, int(s)), floor) for x in img_size]
  else:
      raise Exception(f"Unsupported type of img_size: {type(img_size)}")

  if new_size != img_size:
      print(f'WARNING: --img-size {img_size} must be multiple of max stride {s}, updating to {new_size}')
  return new_size if isinstance(img_size,list) else [new_size]*2


def process_image(path, img_size, stride, half):
  '''Process image before image inference.'''
  try:
    img_src = np.asarray(PIL.Image.open(path))
    assert img_src is not None, f'Invalid image: {path}'
  except Exception as e:
    LOGGER.warning(e)
    return None, None  # Return None for both image and img_src in case of an exception

  image = letterbox(img_src, img_size, stride=stride)[0]

  # Convert
  image = image.transpose((2, 0, 1))  # HWC to CHW
  image = torch.from_numpy(np.ascontiguousarray(image))
  image = image.half() if half else image.float()  # uint8 to fp16/32
  image /= 255  # 0 - 255 to 0.0 - 1.0

  return image, img_src


model = DetectBackend(f"./{checkpoint}.pt", device=device)
stride = model.stride
class_names = load_yaml("./data/coco.yaml")['names']

if half & (device.type != 'cpu'):
  model.model.half()
else:
  model.model.float()
  half = False

img_size:int = 640#@param {type:"integer"}

if device.type != 'cpu':
  model(torch.zeros(1, 3, *img_size).to(device).type_as(next(model.model.parameters())))  # warmup

#@title Run YOLOv6 on an image from a local path. { run: "auto" }
hide_labels: bool = False #@param {type:"boolean"}
hide_conf: bool = False #@param {type:"boolean"}

img_size:int = 640#@param {type:"integer"}

conf_thres: float =.60 #@param {type:"number"}
iou_thres: float =.80 #@param {type:"number"}
max_det:int =  1000#@param {type:"integer"}
agnostic_nms: bool = False #@param {type:"boolean"}

img_size = check_img_size(img_size, s=stride)

img, img_src = process_image(image_path, img_size, stride, half)
img = img.to(device)
if len(img.shape) == 3:
    img = img[None]
    # expand for batch dim
pred_results = model(img)
classes:Optional[List[int]] = None # the classes to keep
det = non_max_suppression(pred_results, conf_thres, iou_thres, classes, agnostic_nms, max_det=max_det)[0]

gn = torch.tensor(img_src.shape)[[1, 0, 1, 0]]  # normalization gain whwh
img_ori = img_src.copy()
if len(det):
  det[:, :4] = Inferer.rescale(img.shape[2:], det[:, :4], img_src.shape).round()
  for *xyxy, conf, cls in reversed(det):
      class_num = int(cls)
      label = None if hide_labels else (class_names[class_num] if hide_conf else f'{class_names[class_num]} {conf:.2f}')
      Inferer.plot_box_and_label(img_ori, max(round(sum(img_ori.shape) / 2 * 0.003), 2), xyxy, label, color=Inferer.generate_colors(class_num, True))
PIL.Image.fromarray(img_ori)



### Extract bed coordinates from bed calibration image

In [ ]:
def read_yolo_annotation(annotation_path, image_width, image_height):
    with open(annotation_path, 'r') as file:
        lines = file.readlines()

    bounding_boxes = []
    for line in lines:
        data = line.strip().split()
        class_index = int(data[0])
        x_center, y_center, width, height = map(float, data[1:])
        
        # Convert from normalized coordinates to image coordinates
        x_min = int((x_center - width / 2) * image_width)
        y_min = int((y_center - height / 2) * image_height)
        x_max = int((x_center + width / 2) * image_width)
        y_max = int((y_center + height / 2) * image_height)

        bounding_boxes.append((class_index, x_min, y_min, x_max, y_max))

    return bounding_boxes

# Function to save YOLO annotation file for the bed class only
def save_bed_annotation(annotation_path, det, img_width, img_height, bed_index):
    with open(annotation_path, 'w') as file:
        for *xyxy, conf, cls in reversed(det):
            class_num = int(cls)

            # Check if the detected class is the bed class
            if class_num == bed_index:
                # Calculate XYWH format directly from XYXY
                x_center = (xyxy[0] + xyxy[2]) / 2
                y_center = (xyxy[1] + xyxy[3]) / 2
                width = xyxy[2] - xyxy[0]
                height = xyxy[3] - xyxy[1]

                # Normalize coordinates to [0, 1]
                x_center /= img_width
                y_center /= img_height
                width /= img_width
                height /= img_height

                line = f"{class_num} {x_center} {y_center} {width} {height}\n"
                file.write(line)

# Function to generate output paths dynamically
def generate_output_paths(crop_out_folder, class_name, index):
    class_folder = os.path.join(crop_out_folder, class_name)
    os.makedirs(class_folder, exist_ok=True)
    output_path = os.path.join(class_folder, f'{class_name}_{index}.jpg')
    return output_path

# Replace these paths with your actual folder paths
jpg_output_images_folder = '/Users/emibetroldan/Downloads/Science_Fair/fotos_out/yolo_out/jpg_files'
txt_output_annotations_folder = '/Users/emibetroldan/Downloads/Science_Fair/fotos_out/yolo_out/txt_files'
crop_out_folder = '/Users/emibetroldan/Downloads/Science_Fair/fotos_out/crop_out'

os.makedirs(jpg_output_images_folder, exist_ok=True)
os.makedirs(txt_output_annotations_folder, exist_ok=True)

# ...

# Save image to JPEG
output_image_path = os.path.join(jpg_output_images_folder, 'output_image.jpg')
cv2.imwrite(output_image_path, cv2.cvtColor(img_ori, cv2.COLOR_RGB2BGR))

# Save YOLO annotation file for the bed class only
output_annotation_path = os.path.join(txt_output_annotations_folder, 'output_image.txt')
save_bed_annotation(output_annotation_path, det, img_ori.shape[1], img_ori.shape[0], 59)

# Display the processed image
PIL.Image.fromarray(cv2.cvtColor(img_ori, cv2.COLOR_BGR2RGB))

# Read YOLO annotation
bounding_boxes = read_yolo_annotation(output_annotation_path, img_ori.shape[1], img_ori.shape[0])

# Crop and save regions only if the detected class is a bed
for i, (class_index, x_min, y_min, x_max, y_max) in enumerate(bounding_boxes):
    class_name = class_names[class_index]
    
    # Check if the detected class is a bed (replace 'bed_index' with the actual index of the bed class)
    if class_index == 59:
        output_path = generate_output_paths(crop_out_folder, class_name, i)
        cropped_region = img_ori[y_min:y_max, x_min:x_max]
        cv2.imwrite(output_path, cv2.cvtColor(cropped_region, cv2.COLOR_BGR2RGB))


In [ ]:
def calculate_coordinates_from_yolo_annotation(annotation_path, image_width, image_height):
    with open(annotation_path, 'r') as file:
        lines = file.readlines()

    bounding_boxes = []
    for line in lines:
        data = line.strip().split()
        class_index = int(data[0])
        x_center, y_center, width, height = map(float, data[1:])
        
        # Convert from normalized coordinates to image coordinates
        x_min = int((x_center - width / 2) * image_width)
        y_min = int((y_center - height / 2) * image_height)
        x_max = int((x_center + width / 2) * image_width)
        y_max = int((y_center + height / 2) * image_height)

        bounding_boxes.append((class_index, x_min, y_min, x_max, y_max))

    return bounding_boxes

# Replace these paths with your actual file paths
annotation_path = '/Users/emibetroldan/Downloads/Science_Fair/fotos_out/yolo_out/txt_files/output_image.txt'
image_width = 640  # Replace with the actual width of your image
image_height = 480  # Replace with the actual height of your image

# Read YOLO annotation and calculate coordinates
bounding_boxes = calculate_coordinates_from_yolo_annotation(annotation_path, image_width, image_height)

# Print calculated bounding box coordinates
for i, (class_index, x_min, y_min, x_max, y_max) in enumerate(bounding_boxes):
    print(f"Bounding Box {i + 1}: Class {class_names[class_index]}, Coordinates (x_min, y_min, x_max, y_max): ({x_min}, {y_min}, {x_max}, {y_max})")

bed_coords = (x_min, y_min, x_max, y_max)


In [ ]:
# Start webcam
cap = cv2.VideoCapture(1)
cap.set(3, 640)
cap.set(4, 480)

# Model
model = YOLO("yolo-Weights/yolov8n.pt")

# Object class
classNames = ["person"]

# Permanent boundary box for bed
bed_box_coords = (bed_coords)

fall_alert_time = 0
out_of_bed_alert_time = 0

fall_alert_interval = 30  # seconds
out_of_bed_alert_interval = 30  # seconds

while True:
    success, img = cap.read()
    results = model(img, stream=True)

    person_detected = False
    person_box = None

    # Draw the permanent boundary box for the bed
    bed_color = (255, 255, 255)
    bed_thickness = 2
    cv2.rectangle(img, (bed_box_coords[0], bed_box_coords[1]), (bed_box_coords[2], bed_box_coords[3]), bed_color, bed_thickness)

    # Mark the boundary box of the bed with the word "Bed"
    bed_text = "Bed"
    bed_text_color = (255, 255, 255)  # White text
    bed_text_font = cv2.FONT_HERSHEY_SIMPLEX
    bed_text_scale = 1
    bed_text_thickness = 2

    # Get the size of the text box
    (text_width, text_height), baseline = cv2.getTextSize(bed_text, bed_text_font, bed_text_scale, bed_text_thickness)

    # Calculate the position to the right of the boundary box
    bed_text_x = bed_box_coords[2] + 10  # Adjust the offset as needed
    bed_text_y = (bed_box_coords[1] + bed_box_coords[3] - text_height) // 2

    # Draw the outlined text
    bed_text_border_thickness = 5
    cv2.putText(img, bed_text, (bed_text_x, bed_text_y), bed_text_font, bed_text_scale, (0, 0, 0), bed_text_border_thickness + 2, cv2.LINE_AA)
    cv2.putText(img, bed_text, (bed_text_x, bed_text_y), bed_text_font, bed_text_scale, bed_text_color, bed_text_thickness, cv2.LINE_AA)

    # Coordinates
    for r in results:
        boxes = r.boxes

        for box in boxes:
            # Bounding box
            x1, y1, x2, y2 = box.xyxy[0]
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)  # Convert to int values

            # Confidence
            confidence = math.ceil((box.conf[0] * 100)) / 100

            # Ensure class index is within the valid range
            class_index = int(box.cls[0])
            if class_index < 0 or class_index >= len(classNames):
                continue  # Skip if index is out of range

            # Show bounding box only if confidence is 60 or more and class is "person"
            if confidence >= 0.6 and classNames[class_index] == "person":
                # Put box in the camera
                color = (255, 0, 255)  # Pink for person
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)

                # Class name
                cls = classNames[class_index]

                # Check for person detection
                if cls == "person":
                    person_detected = True
                    person_box = (x1, y1, x2, y2)

                # Display class name, confidence, and overlap area with enhanced styling
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 1.2  # Adjust the font size here
                font_thickness = 2

                # Draw class name with neon color and bold text with black border
                text_color = (255, 192, 203)  # Pink for person
                border_thickness = 5
                cv2.putText(img, cls, (x1, y1 - 10), font, font_scale, (0, 0, 0), border_thickness + 2, cv2.LINE_AA)
                cv2.putText(img, cls, (x1, y1 - 10), font, font_scale, text_color, font_thickness, cv2.LINE_AA)

                # Display confidence percentage with black border
                cv2.putText(img, f"{confidence*100:.2f}%", (x1, y1 + 25), font, font_scale, (0, 0, 0), border_thickness, cv2.LINE_AA)
                cv2.putText(img, f"{confidence*100:.2f}%", (x1, y1 + 25), font, font_scale, text_color, font_thickness, cv2.LINE_AA)

    # Check the position of the person relative to the bed boundary box
    if person_detected:
        px1, py1, px2, py2 = person_box

        # Check if person is in bed, out of bed, or getting out of bed
        if px1 >= bed_box_coords[0] and py1 >= bed_box_coords[1] and px2 <= bed_box_coords[2] and py2 <= bed_box_coords[3]:
            status = "Patient is in bed."
            status_color = (0, 255, 0)  # Green
        elif px1 > bed_box_coords[2] or px2 < bed_box_coords[0] or py1 > bed_box_coords[3] or py2 < bed_box_coords[1]:
            status = "Alert! Patient is out of the bed!"
            status_color = (0, 0, 255)  # Red

            # Check if it's time to send a fall alert or if a fall just occurred
            if time.time() - fall_alert_time >= fall_alert_interval or fall_alert_time == 0:
                # Send alert email for fallen patient
                caregiver_name = "Ms. Emibet"
                mail = mt.MailFromTemplate(
                    sender=mt.Address(email="mailtrap@demomailtrap.com", name="Mailtrap Test"),
                    to=[mt.Address(email="sciencefair2024@yahoo.com")],
                    template_uuid="8302914b-6dc1-4c05-89c3-1b6260803268",
                    template_variables={"Caregiver_name": caregiver_name}
                )
                client = mt.MailtrapClient(token="1467c284eb77bd4de72abdac13ab416b")
                client.send(mail)

                fall_alert_time = time.time()  # Update last alert time

        else:
            status = "Grandma is getting out of bed!"
            status_color = (0, 165, 255)  # Orange

            # Check if it's time to send an out-of-bed alert
            if time.time() - out_of_bed_alert_time >= out_of_bed_alert_interval:
                # Send alert email for going out of bed
                caregiver_name = "Ms. Emibet"
                mail = mt.MailFromTemplate(
                    sender=mt.Address(email="mailtrap@demomailtrap.com", name="Mailtrap Test"),
                    to=[mt.Address(email="sciencefair2024@yahoo.com")],
                    template_uuid="0a471fa2-94dd-4d88-92c2-9bdab62ef810",
                    template_variables={"Caregiver_name": caregiver_name}
                )
                client = mt.MailtrapClient(token="1467c284eb77bd4de72abdac13ab416b")
                client.send(mail)

                out_of_bed_alert_time = time.time()  # Update last alert time

        # Display the status with enhanced styling in the bottom left corner
        cv2.putText(img, f"{status}", (10, 480 - 10), font, 1, (0, 0, 0), 5, cv2.LINE_AA)
        cv2.putText(img, f"{status}", (10, 480 - 10), font, 1, status_color, 2, cv2.LINE_AA)

    cv2.imshow('Webcam', img)
    if cv2.waitKey(1) == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
